### Finding Cluster

In [158]:
import pandas as pd
import os
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import OPTICS

In [159]:
folder_path = "Stocks"

dfs=[]
for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        df = df.pivot_table(index=[], columns='Date', values=file[:-4])
        dfs.append(df)

df=pd.concat(dfs)
df=df.pct_change(axis=1)
df=df.drop(['2023-01-03'],axis=1)

In [160]:
#Apply PCA
scaler = StandardScaler()
returns_scaled = scaler.fit_transform(df)
pca = PCA(n_components=15) # Adjust the number of components as needed
pca_data = pca.fit_transform(returns_scaled)



In [161]:
optics = OPTICS(min_samples=2)  # Adjust parameters as needed
clusters = optics.fit_predict(pca_data)  # Fit OPTICS on PCA data
pca_df = pd.DataFrame(data=pca_data, columns=[f'PC{i+1}' for i in range(pca_data.shape[1])])
pca_df['Cluster'] = clusters  # Add cluster label to DataFrame
pca_df.index=df.index

In [162]:
clusters={}
for i, row in pca_df.iterrows():
    if row['Cluster'] not in clusters:
        clusters[row['Cluster']]=[i]
    else:
        clusters[row['Cluster']].append(i)

clusters

{0.0: ['AAPL', 'MSFT'],
 7.0: ['ABBV', 'JNJ'],
 -1.0: ['ABT',
  'ACN',
  'AIG',
  'AMGN',
  'AVGO',
  'AXP',
  'BA',
  'BKNG',
  'BLK',
  'CHTR',
  'CMCSA',
  'COF',
  'COST',
  'CSCO',
  'DIS',
  'DOW',
  'GE',
  'IBM',
  'INTC',
  'LLY',
  'MDT',
  'MMM',
  'MRK',
  'NEE',
  'NFLX',
  'NKE',
  'NVDA',
  'ORCL',
  'PYPL',
  'SBUX',
  'SCHW',
  'SPG',
  'TGT',
  'TMUS',
  'TSLA',
  'UNP',
  'USB',
  'WMT'],
 18.0: ['ADBE', 'CRM'],
 17.0: ['AMZN', 'META'],
 12.0: ['BAC', 'WFC'],
 10.0: ['BK', 'JPM', 'MET'],
 8.0: ['BMY', 'PFE'],
 11.0: ['C', 'GS', 'MS'],
 15.0: ['CAT', 'EMR'],
 3.0: ['CL', 'KO', 'MCD', 'MDLZ', 'PEP', 'PG'],
 20.0: ['COP', 'CVX', 'XOM'],
 23.0: ['CVS', 'UNH'],
 16.0: ['DHR', 'TMO'],
 6.0: ['DUK', 'EXC', 'SO'],
 22.0: ['F', 'GM'],
 14.0: ['FDX', 'UPS'],
 13.0: ['GD', 'LMT', 'RTX'],
 21.0: ['GOOG', 'GOOGL'],
 9.0: ['HD', 'LOW'],
 1.0: ['HON', 'LIN'],
 2.0: ['MA', 'V'],
 4.0: ['MO', 'PM'],
 19.0: ['QCOM', 'TXN'],
 5.0: ['T', 'VZ']}

In [163]:
del clusters[-1]
clusters

{0.0: ['AAPL', 'MSFT'],
 7.0: ['ABBV', 'JNJ'],
 18.0: ['ADBE', 'CRM'],
 17.0: ['AMZN', 'META'],
 12.0: ['BAC', 'WFC'],
 10.0: ['BK', 'JPM', 'MET'],
 8.0: ['BMY', 'PFE'],
 11.0: ['C', 'GS', 'MS'],
 15.0: ['CAT', 'EMR'],
 3.0: ['CL', 'KO', 'MCD', 'MDLZ', 'PEP', 'PG'],
 20.0: ['COP', 'CVX', 'XOM'],
 23.0: ['CVS', 'UNH'],
 16.0: ['DHR', 'TMO'],
 6.0: ['DUK', 'EXC', 'SO'],
 22.0: ['F', 'GM'],
 14.0: ['FDX', 'UPS'],
 13.0: ['GD', 'LMT', 'RTX'],
 21.0: ['GOOG', 'GOOGL'],
 9.0: ['HD', 'LOW'],
 1.0: ['HON', 'LIN'],
 2.0: ['MA', 'V'],
 4.0: ['MO', 'PM'],
 19.0: ['QCOM', 'TXN'],
 5.0: ['T', 'VZ']}

## Test Cluster

### Engle Granger Test

In [168]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller

def is_I1(series):
    """Check if a series is I(1): non-stationary in levels, stationary in first differences."""
    series = series.dropna()
    
    # ADF on levels
    p_level = adfuller(series)[1]
    if p_level < 0.05:
        return False  # stationary at level → not I(1)
    
    # ADF on first difference
    p_diff = adfuller(series.diff().dropna())[1]
    if p_diff < 0.05:
        return True   # stationary after differencing → I(1)
    
    return False      # could be I(2) or higher

def engle_granger_test(stock1, stock2, df):
    """Engle–Granger cointegration test between two stock price series."""
    x = df.loc[stock1, :].dropna()
    y = df.loc[stock2, :].dropna()
    
    model = sm.OLS(x, sm.add_constant(y)).fit()
    residuals = model.resid

    # Step 3: ADF test on residuals
    p_resid = adfuller(residuals)[1]
    if p_resid < 0.05:
        return True  # cointegration detected
    
    return False


In [169]:
result_x

(-13.900229314029213,
 5.752357238796114e-26,
 4,
 746,
 {'1%': -3.439146171679794,
  '5%': -2.865422101274577,
  '10%': -2.568837245865348},
 -3958.2731769012926)

In [170]:
for cluster in clusters:
    stocks=clusters[cluster]
    
    for i in range(len(stocks)):
        for j in range(i+1,len(stocks)):
            if engle_granger_test(stocks[i],stocks[j],df):
                print(f"{stocks[i]} and {stocks[j]} okay")
            

AAPL and MSFT okay
ABBV and JNJ okay
ADBE and CRM okay
AMZN and META okay
BAC and WFC okay
BK and JPM okay
BK and MET okay
JPM and MET okay
BMY and PFE okay
C and GS okay
C and MS okay
GS and MS okay
CAT and EMR okay
CL and KO okay
CL and MCD okay
CL and MDLZ okay
CL and PEP okay
CL and PG okay
KO and MCD okay
KO and MDLZ okay
KO and PEP okay
KO and PG okay
MCD and MDLZ okay
MCD and PEP okay
MCD and PG okay
MDLZ and PEP okay
MDLZ and PG okay
PEP and PG okay
COP and CVX okay
COP and XOM okay
CVX and XOM okay
CVS and UNH okay
DHR and TMO okay
DUK and EXC okay
DUK and SO okay
EXC and SO okay
F and GM okay
FDX and UPS okay
GD and LMT okay
GD and RTX okay
LMT and RTX okay
GOOG and GOOGL okay
HD and LOW okay
HON and LIN okay
MA and V okay
MO and PM okay
QCOM and TXN okay
T and VZ okay


ADF Test on AAPL and MSFT:

If the test statistic for either AAPL or MSFT is less (more negative) than the critical values, or if the p-value is below 0.05, you can conclude that the respective series is stationary (no unit root).
ADF Test on Residuals:

If the residuals are stationary, it implies that while AAPL and MSFT may each be non-stationary on their own, they move together in a long-term relationship (cointegration).


In [167]:
engle_granger_test('JNJ','VZ')

TypeError: engle_granger_test() missing 1 required positional argument: 'df'